# Dataset 2 - Cityscapes Dataframe Generation

This notebook performs **external/generalization evaluation dataframe generation only**.

The model was fine-tuned on BDD100K. Cityscapes is used only as a second evaluation dataset. The notebook therefore:

1. loads and validates the selected Cityscapes images and polygon annotations,
2. converts Cityscapes annotations into evaluation-compatible ground truth for the four established BDD100K tasks,
3. builds one task-level evaluation dataframe,
4. runs the same Qwen3-VL base model and the same model with the trained LoRA adapter enabled,
5. saves aligned base and fine-tuned JSONL dataframes.



Important Dataset 2 limitations are preserved explicitly:
- Cityscapes polygon annotations do not provide a BDD-compatible weather label, so weather ground truth is left unavailable rather than fabricated.
- Hazard ground truth uses the agreed conservative Cityscapes annotation-based VRU proxy.


**Configuration**

In [ ]:
from pathlib import Path
from contextlib import nullcontext
from datetime import datetime, timezone
import gc
import json
import time

import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_DIR = Path(r"C:\Users\becker\Documents\Thesis")

CITYSCAPES_IMAGE_DIR = PROJECT_DIR / "dataset" / "CityScapes" / "images"
CITYSCAPES_LABEL_DIR = PROJECT_DIR / "dataset" / "CityScapes" / "labels"
OUTPUT_DIR = PROJECT_DIR / "dataset" / "CityScapes"

BASE_OUTPUT_PATH = OUTPUT_DIR / "cityscapes_base_dataframe.jsonl"
FINETUNED_OUTPUT_PATH = OUTPUT_DIR / "cityscapes_finetuned_dataframe.jsonl"

BASE_MODEL_NAME = "Qwen/Qwen3-VL-4B-Instruct"
LORA_MODEL_PATH = PROJECT_DIR / "qwen3vl_bdd100k_lora" / "final_model"

EXPECTED_IMAGE_COUNT = 267
MIN_PIXELS = 256 * 256
MAX_PIXELS = 384 * 384
MAX_NEW_TOKENS = 64
CHECKPOINT_EVERY = 25
PIPELINE_VERSION = "cityscapes_external_eval_v2"

# Exact BDD100K evaluation vocabulary.
OBJECT_LABELS = [
    "car", "truck", "bus", "person", "rider",
    "bicycle", "motorcycle", "train",
    "traffic light", "traffic sign",
]

WEATHER_LABELS = [
    "clear", "overcast", "partly cloudy",
    "rainy", "snowy", "foggy",
]

SCENE_LABELS = [
    "city street", "highway", "residential",
    "parking lot", "tunnel", "gas station",
]

# Dataset-level context supported by the selected Frankfurt Cityscapes subset.
CITYSCAPES_SCENE = "city street"
CITYSCAPES_TIME_OF_DAY = "daytime"
CITYSCAPES_WEATHER = None  # unavailable

# Exact task prompts used in the established BDD100K evaluation setup.
TASK_DEFINITIONS = {
    "multilabel_object_recognition": {
        "task": "Multilabel Object Recognition",
        "instruction": (
            "Identify all visible traffic-object classes in the image. "
            "Return only the class names as a comma-separated list. "
            "Use only these labels: car, truck, bus, person, rider, bicycle, "
            "motorcycle, train, traffic light, traffic sign."
        ),
        "operational_definition": None,
    },
    "environmental_scene_understanding": {
        "task": "Environmental Scene Understanding",
        "instruction": (
            "Identify the weather condition and scene type shown in the image. "
            "Return only in this format: weather: <label>; scene: <label>. "
            "Weather labels: clear, overcast, partly cloudy, rainy, snowy, foggy. "
            "Scene labels: city street, highway, residential, parking lot, tunnel, gas station."
        ),
        "operational_definition": (
            "Cityscapes supplies a compatible scene-domain label for this subset, "
            "but no BDD-compatible per-image weather annotation. Weather ground truth "
            "is therefore unavailable and must not be scored as if annotated."
        ),
    },
    "hazard_and_anomaly_detection": {
        "task": "Hazard and Anomaly Detection",
        "instruction": (
            "Determine whether the image contains a rule-defined visible potential hazard: "
            "either a sufficiently large vulnerable road user very near the directly drivable area, "
            "or a large close vehicle occupying the central directly drivable area. "
            "Return the decision, hazard type, involved objects, location, severity, and visible evidence."
        ),
        "operational_definition": (
            "Dataset 2 uses a conservative Cityscapes-derived proxy: a positive is assigned "
            "when an annotated vulnerable road user reaches the lower 35% of the image and its "
            "estimated ground-contact point lies inside an annotated road polygon and outside "
            "annotated sidewalk polygons. The close-vehicle branch is not operationalized because "
            "Cityscapes does not provide the same BDD100K hazard/drivable-area annotation structure."
        ),
    },
    "traffic_scene_captioning": {
        "task": "Traffic Scene Captioning",
        "instruction": (
            "Describe the traffic scene in one concise paragraph. Include the weather, time of day, "
            "scene type, and visible traffic-related objects. Do not speculate about details that are not visible."
        ),
        "operational_definition": (
            "Dataset 2 reference captions are deterministic annotation-derived references. "
            "They encode supported scene/time context and visible evaluation classes without inventing weather."
        ),
    },
}

assert CITYSCAPES_IMAGE_DIR.exists(), f"Image directory not found: {CITYSCAPES_IMAGE_DIR}"
assert CITYSCAPES_LABEL_DIR.exists(), f"Label directory not found: {CITYSCAPES_LABEL_DIR}"
assert LORA_MODEL_PATH.exists(), f"LoRA adapter directory not found: {LORA_MODEL_PATH}"
assert CITYSCAPES_SCENE in SCENE_LABELS
assert len(TASK_DEFINITIONS) == 4

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration validated.")
print(f"Expected images: {EXPECTED_IMAGE_COUNT}")
print(f"Base output: {BASE_OUTPUT_PATH}")
print(f"Fine-tuned output: {FINETUNED_OUTPUT_PATH}")


Configuration validated.
Expected images: 267
Base output: C:\Users\becker\Documents\Thesis\dataset\CityScapes\cityscapes_base_dataframe.jsonl
Fine-tuned output: C:\Users\becker\Documents\Thesis\dataset\CityScapes\cityscapes_finetuned_dataframe.jsonl


**Load and Verify Cityscapes Image/Annotation Pairs**

In [ ]:
image_paths = sorted(CITYSCAPES_IMAGE_DIR.glob("*_leftImg8bit.png"))

assert len(image_paths) == EXPECTED_IMAGE_COUNT, (
    f"Expected {EXPECTED_IMAGE_COUNT} images, found {len(image_paths)}."
)

samples = []

for image_path in image_paths:
    image_id = image_path.name.replace("_leftImg8bit.png", "")
    label_path = CITYSCAPES_LABEL_DIR / f"{image_id}_gtFine_polygons.json"

    samples.append({
        "image_id": image_id,
        "image_filename": image_path.name,
        "image_path": str(image_path),
        "label_path": str(label_path),
    })

df_pairs = pd.DataFrame(samples)

assert df_pairs["image_id"].is_unique
assert df_pairs["image_path"].is_unique

missing_labels = [
    path for path in df_pairs["label_path"]
    if not Path(path).exists()
]
assert not missing_labels, (
    f"{len(missing_labels)} annotation files are missing. "
    f"First missing file: {missing_labels[0] if missing_labels else None}"
)

print("Cityscapes file pairing validated.")
print(f"Images: {len(df_pairs)}")
print(f"Unique image IDs: {df_pairs['image_id'].nunique()}")
print("Missing labels: 0")


Cityscapes file pairing validated.
Images: 267
Unique image IDs: 267
Missing labels: 0


**Load and Normalize Native Cityscapes Annotations**

In [ ]:
REQUIRED_ANNOTATION_KEYS = {"imgWidth", "imgHeight", "objects"}

def normalize_cityscapes_label(label):
    label = str(label).strip().lower()

    # Cityscapes polygon files may contain group labels such as 'persongroup'.
    if label.endswith("group"):
        candidate = label[:-5].strip()
        if candidate:
            label = candidate

    aliases = {
        "trafficlight": "traffic light",
        "trafficsign": "traffic sign",
    }
    return aliases.get(label, label)


def normalize_objects(objects):
    normalized = []

    for obj_index, obj in enumerate(objects):
        if not isinstance(obj, dict):
            continue

        # Deleted annotations must not contribute to ground truth.
        if bool(obj.get("deleted", False)):
            continue

        raw_label = obj.get("label")
        if raw_label is None:
            continue

        polygon = obj.get("polygon", [])
        if not isinstance(polygon, list):
            polygon = []

        normalized.append({
            "object_index": obj_index,
            "source_label": str(raw_label),
            "label": normalize_cityscapes_label(raw_label),
            "polygon": polygon,
        })

    return normalized


records = []

for row in df_pairs.itertuples(index=False):
    with open(row.label_path, "r", encoding="utf-8") as f:
        annotation = json.load(f)

    missing_keys = REQUIRED_ANNOTATION_KEYS - set(annotation)
    assert not missing_keys, (
        f"{row.image_id} is missing annotation keys: {sorted(missing_keys)}"
    )

    width = int(annotation["imgWidth"])
    height = int(annotation["imgHeight"])

    assert width > 0 and height > 0
    assert isinstance(annotation["objects"], list)

    records.append({
        "image_id": row.image_id,
        "image_filename": row.image_filename,
        "image_path": row.image_path,
        "label_path": row.label_path,
        "img_width": width,
        "img_height": height,
        "objects": normalize_objects(annotation["objects"]),
    })

df_images = pd.DataFrame(records)

assert len(df_images) == EXPECTED_IMAGE_COUNT
assert df_images["image_id"].is_unique
assert df_images["objects"].apply(lambda x: isinstance(x, list)).all()

print("Native annotations loaded and normalized.")
print(f"Images: {len(df_images)}")
print(f"Total active annotations: {sum(len(x) for x in df_images['objects'])}")


Native annotations loaded and normalized.
Images: 267
Total active annotations: 28339


**Multilabel Object Recognition Ground Truth**

In [ ]:
def build_object_ground_truth(objects):
    present = {
        obj["label"]
        for obj in objects
        if obj.get("label") in OBJECT_LABELS
    }

    classes = [
        label for label in OBJECT_LABELS
        if label in present
    ]

    presence = {
        label: int(label in present)
        for label in OBJECT_LABELS
    }

    return classes, presence


object_results = df_images["objects"].apply(build_object_ground_truth)

df_images["ground_truth_classes"] = object_results.apply(lambda x: x[0])
df_images["object_presence"] = object_results.apply(lambda x: x[1])

for classes, presence in zip(
    df_images["ground_truth_classes"],
    df_images["object_presence"],
):
    assert set(classes).issubset(set(OBJECT_LABELS))
    assert set(presence) == set(OBJECT_LABELS)
    assert all(value in {0, 1} for value in presence.values())

class_occurrence = {
    label: int(sum(presence[label] for presence in df_images["object_presence"]))
    for label in OBJECT_LABELS
}

print("Object-recognition ground truth created.")
print("Image-level class occurrence:")
for label, count in class_occurrence.items():
    print(f"  {label}: {count}")


Object-recognition ground truth created.
Image-level class occurrence:
  car: 260
  truck: 34
  bus: 43
  person: 245
  rider: 130
  bicycle: 188
  motorcycle: 62
  train: 22
  traffic light: 198
  traffic sign: 262


**Environmental Scene Understanding Ground Truth**

In [ ]:
df_images["ground_truth_weather"] = None
df_images["ground_truth_scene"] = CITYSCAPES_SCENE
df_images["time_of_day"] = CITYSCAPES_TIME_OF_DAY

assert df_images["ground_truth_weather"].isna().all()
assert (df_images["ground_truth_scene"] == CITYSCAPES_SCENE).all()
assert (df_images["time_of_day"] == CITYSCAPES_TIME_OF_DAY).all()

print("Environmental ground truth created.")
print(f"Scene: {CITYSCAPES_SCENE}")
print("Weather: unavailable in Cityscapes polygon annotations")
print(f"Time of day context: {CITYSCAPES_TIME_OF_DAY}")


Environmental ground truth created.
Scene: city street
Weather: unavailable in Cityscapes polygon annotations
Time of day context: daytime


**Hazard and Anomaly Ground Truth Proxy**

In [ ]:
VRU_LABELS = {"person", "rider", "bicycle", "motorcycle"}
HAZARD_Y_FRACTION = 0.65
HAZARD_RULE_VERSION = "cityscapes_vru_road_lower35_v1"

def point_in_polygon(point, polygon):
    if not isinstance(polygon, list) or len(polygon) < 3:
        return False

    x, y = point
    inside = False
    j = len(polygon) - 1

    for i in range(len(polygon)):
        try:
            xi, yi = float(polygon[i][0]), float(polygon[i][1])
            xj, yj = float(polygon[j][0]), float(polygon[j][1])
        except (TypeError, ValueError, IndexError):
            j = i
            continue

        intersects = (
            ((yi > y) != (yj > y))
            and (
                x
                < (xj - xi) * (y - yi) / ((yj - yi) + 1e-9) + xi
            )
        )

        if intersects:
            inside = not inside

        j = i

    return inside


def evaluate_cityscapes_hazard(objects, img_height):
    road_polygons = [
        obj["polygon"]
        for obj in objects
        if obj.get("label") == "road"
        and isinstance(obj.get("polygon"), list)
        and len(obj["polygon"]) >= 3
    ]

    sidewalk_polygons = [
        obj["polygon"]
        for obj in objects
        if obj.get("label") == "sidewalk"
        and isinstance(obj.get("polygon"), list)
        and len(obj["polygon"]) >= 3
    ]

    triggering_objects = []

    for obj in objects:
        if obj.get("label") not in VRU_LABELS:
            continue

        polygon = obj.get("polygon", [])
        if not isinstance(polygon, list) or len(polygon) < 3:
            continue

        try:
            xs = [float(point[0]) for point in polygon]
            ys = [float(point[1]) for point in polygon]
        except (TypeError, ValueError, IndexError):
            continue

        y_max = max(ys)

        # Object must reach the lower 35% of the image.
        if y_max < HAZARD_Y_FRACTION * img_height:
            continue

        contact_point = ((min(xs) + max(xs)) / 2.0, y_max)

        on_road = any(
            point_in_polygon(contact_point, road_polygon)
            for road_polygon in road_polygons
        )
        on_sidewalk = any(
            point_in_polygon(contact_point, sidewalk_polygon)
            for sidewalk_polygon in sidewalk_polygons
        )

        if on_road and not on_sidewalk:
            triggering_objects.append({
                "object_index": int(obj["object_index"]),
                "label": obj["label"],
                "bottom_y_fraction": float(y_max / img_height),
                "contact_point": [float(contact_point[0]), float(contact_point[1])],
                "on_road": True,
                "on_sidewalk": False,
            })

    return {
        "ground_truth_anomaly": int(bool(triggering_objects)),
        "rule_features": triggering_objects,
        "rule_version": HAZARD_RULE_VERSION,
    }


hazard_results = df_images.apply(
    lambda row: evaluate_cityscapes_hazard(
        row["objects"],
        row["img_height"],
    ),
    axis=1,
)

df_images["ground_truth_anomaly"] = hazard_results.apply(
    lambda result: result["ground_truth_anomaly"]
)
df_images["rule_features"] = hazard_results.apply(
    lambda result: result["rule_features"]
)
df_images["rule_version"] = hazard_results.apply(
    lambda result: result["rule_version"]
)

assert set(df_images["ground_truth_anomaly"].unique()).issubset({0, 1})

hazard_positive = int(df_images["ground_truth_anomaly"].sum())
hazard_negative = len(df_images) - hazard_positive

print("Hazard proxy ground truth created.")
print(f"Hazard positives: {hazard_positive}")
print(f"Hazard negatives: {hazard_negative}")
print(f"Hazard rate: {hazard_positive / len(df_images) * 100:.2f}%")
print(f"Rule: {HAZARD_RULE_VERSION}")


Hazard proxy ground truth created.
Hazard positives: 20
Hazard negatives: 247
Hazard rate: 7.49%
Rule: cityscapes_vru_road_lower35_v1


**Annotation-Derived Caption References**

In [ ]:
NATURAL_CLASS_NAMES = {
    "car": "cars",
    "truck": "trucks",
    "bus": "buses",
    "person": "people",
    "rider": "riders",
    "bicycle": "bicycles",
    "motorcycle": "motorcycles",
    "train": "trains",
    "traffic light": "traffic lights",
    "traffic sign": "traffic signs",
}

def join_phrases(items):
    if not items:
        return ""
    if len(items) == 1:
        return items[0]
    if len(items) == 2:
        return f"{items[0]} and {items[1]}"
    return ", ".join(items[:-1]) + f", and {items[-1]}"


def build_caption_reference(row):
    # Presence-only wording is intentional: Cityscapes is not being converted into fabricated instance counts for this task.
    visible = [
        NATURAL_CLASS_NAMES[label]
        for label in row["ground_truth_classes"]
    ]

    if visible:
        object_phrase = join_phrases(visible)
        return (
            f"A {row['time_of_day']} {row['ground_truth_scene']} scene "
            f"with visible {object_phrase}."
        )

    return (
        f"A {row['time_of_day']} {row['ground_truth_scene']} scene "
        "with no traffic-object classes from the evaluation vocabulary annotated."
    )


df_images["reference_caption"] = df_images.apply(
    build_caption_reference,
    axis=1,
)
df_images["caption_reference_type"] = "annotation_derived"

assert df_images["reference_caption"].astype(str).str.strip().ne("").all()

print("Caption references created.")
print("Reference type: annotation_derived")
display(
    df_images[
        ["image_id", "ground_truth_classes", "reference_caption"]
    ].head(5)
)


Caption references created.
Reference type: annotation_derived


,image_id,ground_truth_classes,reference_caption
0,frankfurt_000000_000294,"[car, person, traffic sign]","A daytime city street scene with visible cars,..."
1,frankfurt_000000_000576,"[car, rider, bicycle, traffic sign]","A daytime city street scene with visible cars,..."
2,frankfurt_000000_001016,"[car, person, rider, bicycle, traffic sign]","A daytime city street scene with visible cars,..."
3,frankfurt_000000_001236,"[car, person, rider, bicycle, traffic light, t...","A daytime city street scene with visible cars,..."
4,frankfurt_000000_001751,"[car, truck, person, rider, bicycle, traffic l...","A daytime city street scene with visible cars,..."


**Validate All Dataset 2 Ground Truth**

In [ ]:
assert len(df_images) == EXPECTED_IMAGE_COUNT
assert df_images["image_id"].is_unique

for classes in df_images["ground_truth_classes"]:
    assert isinstance(classes, list)
    assert set(classes).issubset(set(OBJECT_LABELS))

for presence in df_images["object_presence"]:
    assert isinstance(presence, dict)
    assert set(presence) == set(OBJECT_LABELS)
    assert all(value in {0, 1} for value in presence.values())

assert df_images["ground_truth_weather"].isna().all()
assert (df_images["ground_truth_scene"] == CITYSCAPES_SCENE).all()
assert set(df_images["ground_truth_anomaly"].unique()).issubset({0, 1})
assert df_images["reference_caption"].astype(str).str.strip().ne("").all()

missing_images = [
    path for path in df_images["image_path"]
    if not Path(path).exists()
]
missing_labels = [
    path for path in df_images["label_path"]
    if not Path(path).exists()
]

assert not missing_images
assert not missing_labels

print("All Dataset 2 ground truth validated successfully.")
print(f"Images: {len(df_images)}")
print(f"Object vocabulary: {len(OBJECT_LABELS)} classes")
print(f"Hazard positives: {int(df_images['ground_truth_anomaly'].sum())}")
print(f"Hazard negatives: {int((df_images['ground_truth_anomaly'] == 0).sum())}")
print(f"Environmental scene GT: {CITYSCAPES_SCENE}")
print("Environmental weather GT: unavailable")
print("Caption references: annotation-derived references")


All Dataset 2 ground truth validated successfully.
Images: 267
Object vocabulary: 10 classes
Hazard positives: 20
Hazard negatives: 247
Environmental scene GT: city street
Environmental weather GT: unavailable
Caption references: annotation-derived references


**Build the 1,068-Row Task Evaluation Dataframe**

In [ ]:
def hazard_ground_truth_text(row):
    if int(row["ground_truth_anomaly"]) == 0:
        return "No Cityscapes proxy hazard is present."

    labels = sorted({
        feature["label"]
        for feature in row["rule_features"]
    })
    involved = ", ".join(labels) if labels else "vulnerable road user"

    return (
        f"A visible {involved} is inside an annotated road region "
        "in the lower portion of the image."
    )


def task_ground_truth(row, task_slug):
    if task_slug == "multilabel_object_recognition":
        return ", ".join(row["ground_truth_classes"])

    if task_slug == "environmental_scene_understanding":
        # Only the supported component is encoded as GT.
        return f"scene: {row['ground_truth_scene']}"

    if task_slug == "hazard_and_anomaly_detection":
        return hazard_ground_truth_text(row)

    if task_slug == "traffic_scene_captioning":
        return row["reference_caption"]

    raise ValueError(f"Unknown task slug: {task_slug}")


evaluation_rows = []

for _, row in df_images.iterrows():
    for task_slug, task_info in TASK_DEFINITIONS.items():
        gt_text = task_ground_truth(row, task_slug)

        evaluation_rows.append({
            "evaluation_id": f"{task_slug}::{row['image_id']}",
            "source_dataset": "Cityscapes",
            "image_id": row["image_id"],
            "image_filename": row["image_filename"],
            "image_path": row["image_path"],
            "label_path": row["label_path"],
            "task": task_info["task"],
            "task_slug": task_slug,
            "instruction": task_info["instruction"],
            "task_operational_definition": task_info["operational_definition"],
            "ground_truth": gt_text,
            "ground_truth_text": gt_text,

            # Task-specific GT fields. Irrelevant fields stay None.
            "ground_truth_classes": (
                row["ground_truth_classes"]
                if task_slug == "multilabel_object_recognition"
                else None
            ),
            "object_presence": (
                row["object_presence"]
                if task_slug == "multilabel_object_recognition"
                else None
            ),
            "ground_truth_weather": (
                row["ground_truth_weather"]
                if task_slug == "environmental_scene_understanding"
                else None
            ),
            "ground_truth_scene": (
                row["ground_truth_scene"]
                if task_slug == "environmental_scene_understanding"
                else None
            ),
            "accepted_weather_labels": (
                WEATHER_LABELS
                if task_slug == "environmental_scene_understanding"
                else None
            ),
            "accepted_scene_labels": (
                SCENE_LABELS
                if task_slug == "environmental_scene_understanding"
                else None
            ),
            "time_of_day": (
                row["time_of_day"]
                if task_slug in {
                    "environmental_scene_understanding",
                    "traffic_scene_captioning",
                }
                else None
            ),
            "ground_truth_anomaly": (
                int(row["ground_truth_anomaly"])
                if task_slug == "hazard_and_anomaly_detection"
                else None
            ),
            "rule_features": (
                row["rule_features"]
                if task_slug == "hazard_and_anomaly_detection"
                else None
            ),
            "rule_version": (
                row["rule_version"]
                if task_slug == "hazard_and_anomaly_detection"
                else None
            ),
            "reference_caption": (
                row["reference_caption"]
                if task_slug == "traffic_scene_captioning"
                else None
            ),
            "reference_type": (
                row["caption_reference_type"]
                if task_slug == "traffic_scene_captioning"
                else None
            ),
            "ground_truth_source": (
                "cityscapes_polygon_annotations"
                if task_slug == "multilabel_object_recognition"
                else "cityscapes_dataset_level_scene_context"
                if task_slug == "environmental_scene_understanding"
                else "cityscapes_polygon_rule_proxy"
                if task_slug == "hazard_and_anomaly_detection"
                else "cityscapes_annotation_derived_reference"
            ),
            "pipeline_version": PIPELINE_VERSION,
        })


task_dataframe = pd.DataFrame(evaluation_rows)

EXPECTED_ROWS = EXPECTED_IMAGE_COUNT * len(TASK_DEFINITIONS)

assert len(task_dataframe) == EXPECTED_ROWS
assert task_dataframe["evaluation_id"].is_unique
assert task_dataframe["image_id"].nunique() == EXPECTED_IMAGE_COUNT

for task_slug in TASK_DEFINITIONS:
    task_rows = task_dataframe[
        task_dataframe["task_slug"] == task_slug
    ]
    assert len(task_rows) == EXPECTED_IMAGE_COUNT
    assert task_rows["instruction"].nunique() == 1
    assert set(task_rows["image_id"]) == set(df_images["image_id"])

print("Task evaluation dataframe created.")
print(f"Images: {EXPECTED_IMAGE_COUNT}")
print(f"Tasks per image: {len(TASK_DEFINITIONS)}")
print(f"Total rows: {len(task_dataframe)}")
print("\nRows per task:")
print(task_dataframe["task"].value_counts())


Task evaluation dataframe created.
Images: 267
Tasks per image: 4
Total rows: 1068

Rows per task:
task
Multilabel Object Recognition        267
Environmental Scene Understanding    267
Hazard and Anomaly Detection         267
Traffic Scene Captioning             267
Name: count, dtype: int64


**Load Qwen3-VL Base Model and Attach the Trained LoRA Adapter**

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import PeftModel
from qwen_vl_utils import process_vision_info

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print("Loading Qwen3-VL processor...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_NAME)

print("Loading Qwen3-VL base model...")
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=model_dtype,
    device_map="auto",
    attn_implementation="sdpa",
)

print("Attaching the BDD100K-trained LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model,
    str(LORA_MODEL_PATH),
)
model.eval()

assert hasattr(model, "disable_adapter"), (
    "PEFT model does not expose disable_adapter(); base/fine separation cannot be guaranteed."
)
assert model.peft_config, "No LoRA adapter configuration was loaded."

MODEL_DEVICE = next(model.parameters()).device

print("Model setup complete.")
print(f"Input device: {MODEL_DEVICE}")
print("Base condition: LoRA adapter disabled")
print("Fine-tuned condition: LoRA adapter enabled")


Loading Qwen3-VL processor...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen3-VL base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Attaching the BDD100K-trained LoRA adapter...
Model setup complete.
Input device: cuda:0
Base condition: LoRA adapter disabled
Fine-tuned condition: LoRA adapter enabled


**Shared Multimodal Inference Functions**

In [ ]:
def prepare_model_inputs(row):
    image_path = str(row["image_path"])
    instruction = str(row["instruction"])

    if not Path(image_path).exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path,
                    "min_pixels": MIN_PIXELS,
                    "max_pixels": MAX_PIXELS,
                },
                {
                    "type": "text",
                    "text": instruction,
                },
            ],
        }
    ]

    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = {
        key: (
            value.to(MODEL_DEVICE)
            if isinstance(value, torch.Tensor)
            else value
        )
        for key, value in inputs.items()
    }

    return inputs


def generate_prediction(inputs, use_adapter):
    #Generate raw text with the base or LoRA-enabled model condition.
    prompt_length = int(inputs["input_ids"].shape[1])

    adapter_context = (
        nullcontext()
        if use_adapter
        else model.disable_adapter()
    )

    start_time = time.perf_counter()

    with torch.inference_mode():
        with adapter_context:
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
            )

    latency_seconds = time.perf_counter() - start_time
    generated_only = generated_ids[:, prompt_length:]

    output_text = processor.batch_decode(
        generated_only,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    if not output_text:
        raise ValueError("Model produced an empty output.")

    return {
        "predicted_output": output_text,
        "input_token_count": prompt_length,
        "output_token_count": int(generated_only.shape[1]),
        "latency_seconds": float(latency_seconds),
    }


print("Shared inference functions defined.")


Shared inference functions defined.


**Four-Task Sanity Check Before the Full Run**

In [ ]:
sanity_records = []

for task_slug in TASK_DEFINITIONS:
    row = task_dataframe[
        task_dataframe["task_slug"] == task_slug
    ].iloc[0]

    # Prepare once so base and fine-tuned conditions receive the same tensors.
    inputs = prepare_model_inputs(row)

    base_result = generate_prediction(
        inputs,
        use_adapter=False,
    )
    fine_result = generate_prediction(
        inputs,
        use_adapter=True,
    )

    sanity_records.append({
        "task": row["task"],
        "evaluation_id": row["evaluation_id"],
        "ground_truth": row["ground_truth"],
        "base_output": base_result["predicted_output"],
        "finetuned_output": fine_result["predicted_output"],
    })

sanity_df = pd.DataFrame(sanity_records)

assert len(sanity_df) == len(TASK_DEFINITIONS)
assert sanity_df["base_output"].astype(str).str.strip().ne("").all()
assert sanity_df["finetuned_output"].astype(str).str.strip().ne("").all()

print("Four-task sanity check passed.")
display(sanity_df)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Four-task sanity check passed.


,task,evaluation_id,ground_truth,base_output,finetuned_output
0,Multilabel Object Recognition,multilabel_object_recognition::frankfurt_00000...,"car, person, traffic sign","car, person, traffic sign, traffic light","car, person, traffic sign, bike, truck, bus, m..."
1,Environmental Scene Understanding,environmental_scene_understanding::frankfurt_0...,scene: city street,weather: clear; scene: city street,weather: clear; scene: city street
2,Hazard and Anomaly Detection,hazard_and_anomaly_detection::frankfurt_000000...,No Cityscapes proxy hazard is present.,Decision: No\n\nReasoning:\n- The image shows ...,Potential hazards: 1 pedestrian(s)/rider(s) ne...
3,Traffic Scene Captioning,traffic_scene_captioning::frankfurt_000000_000294,"A daytime city street scene with visible cars,...",The scene depicts a quiet urban street interse...,This is a driving scene captured during daytim...


**Generate the Base-Model Cityscapes Dataframe**

In [ ]:
INFERENCE_DEFAULTS = {
    "predicted_output": None,
    "eval_status": "Pending",
    "generation_error": None,
    "input_token_count": None,
    "output_token_count": None,
    "max_new_tokens": MAX_NEW_TOKENS,
    "latency_seconds": None,
    "inference_timestamp_utc": None,
}


def initialize_output_dataframe(model_variant, model_reference):
    output_df = task_dataframe.copy(deep=True)

    for column, default in INFERENCE_DEFAULTS.items():
        output_df[column] = default

    output_df["model_variant"] = model_variant
    output_df["model_reference"] = model_reference

    return output_df


def load_or_initialize_output(path, model_variant, model_reference):
    path = Path(path)

    if not path.exists():
        return initialize_output_dataframe(
            model_variant,
            model_reference,
        )

    output_df = pd.read_json(
        path,
        orient="records",
        lines=True,
    )

    required = {
        "evaluation_id",
        "instruction",
        "pipeline_version",
        "eval_status",
        "predicted_output",
    }
    missing = required - set(output_df.columns)
    if missing:
        raise RuntimeError(
            f"Existing checkpoint is incompatible: {path}\n"
            f"Missing columns: {sorted(missing)}"
        )

    if len(output_df) != len(task_dataframe):
        raise RuntimeError(
            f"Existing checkpoint has {len(output_df)} rows; "
            f"current pipeline requires {len(task_dataframe)}."
        )

    if output_df["evaluation_id"].tolist() != task_dataframe["evaluation_id"].tolist():
        raise RuntimeError(
            f"Existing checkpoint evaluation IDs do not match: {path}"
        )

    if output_df["instruction"].tolist() != task_dataframe["instruction"].tolist():
        raise RuntimeError(
            f"Existing checkpoint prompts do not match: {path}"
        )

    if not (output_df["pipeline_version"] == PIPELINE_VERSION).all():
        raise RuntimeError(
            f"Existing checkpoint was produced by another pipeline version: {path}"
        )

    output_df["model_variant"] = model_variant
    output_df["model_reference"] = model_reference

    return output_df


def save_output(dataframe, path):
    dataframe.to_json(
        path,
        orient="records",
        lines=True,
        force_ascii=False,
    )


def run_inference_dataframe(dataframe, output_path, use_adapter, description):
    processed_since_save = 0

    for idx in tqdm(
        range(len(dataframe)),
        total=len(dataframe),
        desc=description,
    ):
        if dataframe.at[idx, "eval_status"] == "Completed":
            continue

        row = dataframe.iloc[idx]

        try:
            inputs = prepare_model_inputs(row)
            result = generate_prediction(
                inputs,
                use_adapter=use_adapter,
            )

            for key, value in result.items():
                dataframe.at[idx, key] = value

            dataframe.at[idx, "eval_status"] = "Completed"
            dataframe.at[idx, "generation_error"] = None
            dataframe.at[idx, "inference_timestamp_utc"] = (
                datetime.now(timezone.utc).isoformat()
            )

        except Exception as exc:
            dataframe.at[idx, "predicted_output"] = None
            dataframe.at[idx, "eval_status"] = "Failed"
            dataframe.at[idx, "generation_error"] = (
                f"{type(exc).__name__}: {exc}"
            )

        processed_since_save += 1

        if processed_since_save % CHECKPOINT_EVERY == 0:
            save_output(dataframe, output_path)

    save_output(dataframe, output_path)
    return dataframe


base_dataframe = load_or_initialize_output(
    BASE_OUTPUT_PATH,
    model_variant="base",
    model_reference=BASE_MODEL_NAME,
)

base_dataframe = run_inference_dataframe(
    base_dataframe,
    BASE_OUTPUT_PATH,
    use_adapter=False,
    description="Cityscapes — Base Model",
)

print("Base-model inference pass finished.")
print(base_dataframe["eval_status"].value_counts(dropna=False))
print(f"Saved to: {BASE_OUTPUT_PATH}")


Cityscapes — Base Model:   0%|          | 0/1068 [00:00<?, ?it/s]

Base-model inference pass finished.
eval_status
Completed    1068
Name: count, dtype: int64
Saved to: C:\Users\becker\Documents\Thesis\dataset\CityScapes\cityscapes_base_dataframe.jsonl


**Generate the Fine-Tuned Cityscapes Dataframe**

In [ ]:
finetuned_dataframe = load_or_initialize_output(
    FINETUNED_OUTPUT_PATH,
    model_variant="fine_tuned",
    model_reference=f"{BASE_MODEL_NAME} + LoRA:{LORA_MODEL_PATH}",
)

finetuned_dataframe = run_inference_dataframe(
    finetuned_dataframe,
    FINETUNED_OUTPUT_PATH,
    use_adapter=True,
    description="Cityscapes — Fine-Tuned Model",
)

print("Fine-tuned inference pass finished.")
print(finetuned_dataframe["eval_status"].value_counts(dropna=False))
print(f"Saved to: {FINETUNED_OUTPUT_PATH}")


Cityscapes — Fine-Tuned Model:   0%|          | 0/1068 [00:00<?, ?it/s]

Fine-tuned inference pass finished.
eval_status
Completed    1068
Name: count, dtype: int64
Saved to: C:\Users\becker\Documents\Thesis\dataset\CityScapes\cityscapes_finetuned_dataframe.jsonl


**Final Alignment Validation and Save**

In [ ]:
for dataframe, name in [
    (base_dataframe, "Base"),
    (finetuned_dataframe, "Fine-tuned"),
]:
    assert len(dataframe) == EXPECTED_ROWS, (
        f"{name} dataframe has {len(dataframe)} rows; expected {EXPECTED_ROWS}."
    )
    assert dataframe["evaluation_id"].is_unique, (
        f"{name} dataframe contains duplicate evaluation IDs."
    )
    assert dataframe["evaluation_id"].tolist() == task_dataframe["evaluation_id"].tolist()
    assert dataframe["instruction"].tolist() == task_dataframe["instruction"].tolist()
    assert dataframe["task_slug"].tolist() == task_dataframe["task_slug"].tolist()

assert (
    base_dataframe["evaluation_id"].tolist()
    == finetuned_dataframe["evaluation_id"].tolist()
)
assert (
    base_dataframe["ground_truth"].fillna("").tolist()
    == finetuned_dataframe["ground_truth"].fillna("").tolist()
)

base_incomplete = int(
    base_dataframe["eval_status"].ne("Completed").sum()
)
fine_incomplete = int(
    finetuned_dataframe["eval_status"].ne("Completed").sum()
)

if base_incomplete or fine_incomplete:
    raise RuntimeError(
        "Dataframe generation is not complete."
        "to retry unfinished rows. "
        f"Base unfinished: {base_incomplete}; "
        f"Fine-tuned unfinished: {fine_incomplete}."
    )

assert base_dataframe["predicted_output"].astype(str).str.strip().ne("").all()
assert finetuned_dataframe["predicted_output"].astype(str).str.strip().ne("").all()

# Final writes after all checks pass.
save_output(base_dataframe, BASE_OUTPUT_PATH)
save_output(finetuned_dataframe, FINETUNED_OUTPUT_PATH)

print("Dataset 2 dataframe generation completed successfully.")
print(f"Rows per dataframe: {EXPECTED_ROWS}")
print(f"Base dataframe: {BASE_OUTPUT_PATH}")
print(f"Fine-tuned dataframe: {FINETUNED_OUTPUT_PATH}")
print("\nTask distribution:")
print(base_dataframe["task"].value_counts())
print(f"\nColumns ({len(base_dataframe.columns)}):")
print(base_dataframe.columns.tolist())


Dataset 2 dataframe generation completed successfully.
Rows per dataframe: 1068
Base dataframe: C:\Users\becker\Documents\Thesis\dataset\CityScapes\cityscapes_base_dataframe.jsonl
Fine-tuned dataframe: C:\Users\becker\Documents\Thesis\dataset\CityScapes\cityscapes_finetuned_dataframe.jsonl

Task distribution:
task
Multilabel Object Recognition        267
Environmental Scene Understanding    267
Hazard and Anomaly Detection         267
Traffic Scene Captioning             267
Name: count, dtype: int64

Columns (36):
['evaluation_id', 'source_dataset', 'image_id', 'image_filename', 'image_path', 'label_path', 'task', 'task_slug', 'instruction', 'task_operational_definition', 'ground_truth', 'ground_truth_text', 'ground_truth_classes', 'object_presence', 'ground_truth_weather', 'ground_truth_scene', 'accepted_weather_labels', 'accepted_scene_labels', 'time_of_day', 'ground_truth_anomaly', 'rule_features', 'rule_version', 'reference_caption', 'reference_type', 'ground_truth_source', 'pipel